In [5]:
# Import library
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
from google.colab import drive
drive.mount('/content/drive')

# Memuat dataset CSV
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/dicoding #9/playstore_reviews.csv')

# Tampilkan beberapa baris pertama
df.head()


# Cek nama kolom
print("Kolom yang tersedia:", df.columns)

# Buat kolom 'sentiment' berdasarkan rating
# Misalnya Rating 1-2 = Negatif, 3 = Netral, 4-5 = Positif
def assign_sentiment(rating):
    if rating <= 2:
        return 'negatif'
    elif rating == 3:
        return 'netral'
    else:
        return 'positif'

df['sentiment'] = df['score'].apply(assign_sentiment)

# Filter data untuk sentimen yang valid
sentiments = ['positif', 'negatif', 'netral']
df = df[df['sentiment'].isin(sentiments)]

# Bersihkan data
df = df.dropna(subset=['score'])

# Preprocessing function
def preprocess(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r"\d+", "", text)
    text = text.strip()
    return text

df['cleaned_text'] = df['content'].apply(preprocess)

# TF-IDF Vectorization
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['cleaned_text'])
y = df['sentiment']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = MultinomialNB(alpha=0.5)  # Adjust alpha
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print("\n🎯 Akurasi Testing:", round(accuracy * 100, 2), "%")
print("\n📊 Classification Report:\n", report)

# Inference
contoh_kalimat = "Aplikasi ini sangat jelek dan sering error"
contoh_clean = preprocess(contoh_kalimat)
contoh_vector = vectorizer.transform([contoh_clean])
prediksi = model.predict(contoh_vector)

print("\n🧠 Inference Contoh:")
print("Kalimat:", contoh_kalimat)
print("Hasil Prediksi Sentimen:", prediksi[0])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Kolom yang tersedia: Index(['reviewId', 'userName', 'userImage', 'content', 'score',
       'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent',
       'repliedAt', 'sortOrder', 'appId'],
      dtype='object')

🎯 Akurasi Testing: 70.99 %

📊 Classification Report:
               precision    recall  f1-score   support

     negatif       0.68      0.84      0.75       958
      netral       0.35      0.01      0.03       421
     positif       0.75      0.86      0.80      1120

    accuracy                           0.71      2499
   macro avg       0.59      0.57      0.53      2499
weighted avg       0.65      0.71      0.65      2499


🧠 Inference Contoh:
Kalimat: Aplikasi ini sangat jelek dan sering error
Hasil Prediksi Sentimen: negatif
